# AI401 — Intelligent Multi-Agent and Expert Systems
## Experiment 1: Knowledge Representation — Semantic Network with Inheritance

**Aim:** Represent knowledge as a semantic network of `(subject, relation, object)` triples and answer queries using property inheritance along `is-a` links, including exceptions.

This notebook follows the experiment sheet: store triples, build inheritance chains, check exceptions before inherited values, report `unknown`, and then complete the three exercises: ostrich, frame representation, and multiple inheritance. fileciteturn0file0L9-L24

## 1. Theory

A semantic network represents concepts as nodes and relationships as labelled, directed edges. The `is-a` relation forms a taxonomy. A property can be inherited from an ancestor unless a more specific concept provides an exception.

For example:

`penguin → is-a → bird → is-a → animal`

`bird → can → fly`

`penguin → cannot → fly`

The query procedure searches from the most specific node upward and gives local knowledge priority. The experiment specifically requires exceptions to be checked before inherited values at every level. fileciteturn0file0L15-L24

## 2. Algorithm

1. Store the network as a list of `(subject, relation, object)` triples.
2. Follow `is-a` links repeatedly to construct the inheritance chain.
3. For `query(node, relation)`, inspect each level from the node upward.
4. At each level, check `cannot` before `can`.
5. Return the first value found.
6. Return `unknown` if the complete chain is exhausted. fileciteturn0file0L25-L30

In [1]:
# Semantic Network represented as (subject, relation, object) triples

knowledge_base = [
    ('animal', 'has', 'skin'),
    ('animal', 'needs', 'food'),
    ('bird', 'is-a', 'animal'),
    ('bird', 'can', 'fly'),
    ('bird', 'has', 'feathers'),
    ('fish', 'is-a', 'animal'),
    ('fish', 'can', 'swim'),
    ('shark', 'is-a', 'fish'),
    ('shark', 'has', 'sharp_teeth'),
    ('penguin', 'is-a', 'bird'),
    ('penguin', 'cannot', 'fly'),
    ('penguin', 'can', 'swim'),
]

print("Knowledge Base:")
for triple in knowledge_base:
    print(triple)

Knowledge Base:
('animal', 'has', 'skin')
('animal', 'needs', 'food')
('bird', 'is-a', 'animal')
('bird', 'can', 'fly')
('bird', 'has', 'feathers')
('fish', 'is-a', 'animal')
('fish', 'can', 'swim')
('shark', 'is-a', 'fish')
('shark', 'has', 'sharp_teeth')
('penguin', 'is-a', 'bird')
('penguin', 'cannot', 'fly')
('penguin', 'can', 'swim')


In [2]:
# Build indexes for efficient lookup

parents = {}
facts = {}
exceptions = {}

for subject, relation, obj in knowledge_base:
    if relation == 'is-a':
        parents.setdefault(subject, []).append(obj)
    elif relation == 'cannot':
        exceptions.setdefault((subject, relation.replace('cannot', 'can')), set()).add(obj)
    else:
        facts.setdefault((subject, relation), set()).add(obj)

print("Parents:", parents)
print("Facts:", facts)
print("Exceptions:", exceptions)

Parents: {'bird': ['animal'], 'fish': ['animal'], 'shark': ['fish'], 'penguin': ['bird']}
Facts: {('animal', 'has'): {'skin'}, ('animal', 'needs'): {'food'}, ('bird', 'can'): {'fly'}, ('bird', 'has'): {'feathers'}, ('fish', 'can'): {'swim'}, ('shark', 'has'): {'sharp_teeth'}, ('penguin', 'can'): {'swim'}}
Exceptions: {('penguin', 'can'): {'fly'}}


In [3]:
def inheritance_chain(node):
    """Return the inheritance chain from the node to its ancestors."""
    chain = []
    current = node
    visited = set()

    while current is not None and current not in visited:
        chain.append(current)
        visited.add(current)
        parent_list = parents.get(current, [])
        current = parent_list[0] if parent_list else None

    return chain


for node in ['penguin', 'shark', 'bird']:
    print(f"{node}: {' -> '.join(inheritance_chain(node))}")

penguin: penguin -> bird -> animal
shark: shark -> fish -> animal
bird: bird -> animal


In [4]:
def query(node, relation, target=None):
    """Query an inherited property, giving exceptions priority.

    If target is supplied, return True/False/None.
    Otherwise return the first target/value found.
    """
    for current in inheritance_chain(node):
        # Exceptions must be checked before inherited/local values.
        blocked = exceptions.get((current, relation), set())
        if target is not None and target in blocked:
            return False

        values = facts.get((current, relation), set())
        if target is not None and target in values:
            return True
        if target is None and blocked:
            return ('cannot', sorted(blocked))
        if target is None and values:
            return ('can', sorted(values))

    return None if target is not None else 'unknown'

## 3. Basic Queries and Results

The experiment asks for the knowledge base, answers for queries, and an inheritance chain for at least one node. fileciteturn0file0L31-L34

In [5]:
queries = [
    ('penguin', 'can', 'fly'),
    ('bird', 'can', 'fly'),
    ('shark', 'has', 'skin'),
    ('shark', 'can', 'swim'),
    ('penguin', 'has', 'skin'),
    ('penguin', 'has', 'feathers'),
    ('penguin', 'can', 'run'),
]

for node, relation, target in queries:
    result = query(node, relation, target)
    print(f"query({node!r}, {relation!r}, {target!r}) -> {result}")

print("\nInheritance chain for penguin:")
print(" -> ".join(inheritance_chain('penguin')))

query('penguin', 'can', 'fly') -> False
query('bird', 'can', 'fly') -> True
query('shark', 'has', 'skin') -> True
query('shark', 'can', 'swim') -> True
query('penguin', 'has', 'skin') -> True
query('penguin', 'has', 'feathers') -> True
query('penguin', 'can', 'run') -> None

Inheritance chain for penguin:
penguin -> bird -> animal


### Expected interpretation

- `query('penguin', 'can', 'fly')` returns **False** because `penguin cannot fly` is found before the inherited `bird can fly` fact.
- `query('bird', 'can', 'fly')` returns **True**.
- `query('shark', 'has', 'skin')` returns **True** because `shark → fish → animal` and `animal has skin`.
- An absent property returns **None**, representing `unknown`.

This is non-monotonic/default reasoning: a general default can be overridden by more specific knowledge. The experiment sheet explicitly asks for this observation. fileciteturn0file0L35-L38

# Exercise 1 — Add an Ostrich

The sheet asks to extend the network with an ostrich that is a bird, cannot fly, and can run fast, then verify the queries. fileciteturn0file0L41-L42

In [6]:
# Add ostrich knowledge
knowledge_base.extend([
    ('ostrich', 'is-a', 'bird'),
    ('ostrich', 'cannot', 'fly'),
    ('ostrich', 'can', 'run_fast'),
])

# Rebuild indexes after modifying the knowledge base
parents = {}
facts = {}
exceptions = {}

for subject, relation, obj in knowledge_base:
    if relation == 'is-a':
        parents.setdefault(subject, []).append(obj)
    elif relation == 'cannot':
        exceptions.setdefault((subject, 'can'), set()).add(obj)
    else:
        facts.setdefault((subject, relation), set()).add(obj)

print("Ostrich chain:", " -> ".join(inheritance_chain('ostrich')))
print("ostrich can fly?     ", query('ostrich', 'can', 'fly'))
print("ostrich can run_fast?", query('ostrich', 'can', 'run_fast'))
print("ostrich has feathers?", query('ostrich', 'has', 'feathers'))
print("ostrich has skin?    ", query('ostrich', 'has', 'skin'))

Ostrich chain: ostrich -> bird -> animal
ostrich can fly?      False
ostrich can run_fast? True
ostrich has feathers? True
ostrich has skin?     True


### Observation — Ostrich

The ostrich inherits general bird properties such as feathers, but its local `cannot fly` exception overrides the bird-level `can fly` default. Its local `can run_fast` property is available directly. This demonstrates that specific knowledge has priority over inherited defaults.

# Exercise 2 — Frame Representation

The same knowledge is now represented using a Python dictionary per concept. The `ako` slot means **a-kind-of**, corresponding to the `is-a` relationship. The experiment sheet explicitly asks for this representation and an inheritance lookup. fileciteturn0file0L43-L44

In [7]:
frames = {
    'animal': {
        'has': 'skin',
        'needs': 'food'
    },
    'bird': {
        'ako': 'animal',
        'can': 'fly',
        'has': 'feathers'
    },
    'fish': {
        'ako': 'animal',
        'can': 'swim'
    },
    'shark': {
        'ako': 'fish',
        'has': 'sharp_teeth'
    },
    'penguin': {
        'ako': 'bird',
        'cannot': 'fly',
        'can': 'swim'
    },
    'ostrich': {
        'ako': 'bird',
        'cannot': 'fly',
        'can': 'run_fast'
    }
}

for concept, slots in frames.items():
    print(concept, '=>', slots)

animal => {'has': 'skin', 'needs': 'food'}
bird => {'ako': 'animal', 'can': 'fly', 'has': 'feathers'}
fish => {'ako': 'animal', 'can': 'swim'}
shark => {'ako': 'fish', 'has': 'sharp_teeth'}
penguin => {'ako': 'bird', 'cannot': 'fly', 'can': 'swim'}
ostrich => {'ako': 'bird', 'cannot': 'fly', 'can': 'run_fast'}


In [8]:
def frame_chain(concept):
    chain = []
    visited = set()
    current = concept

    while current is not None and current not in visited:
        chain.append(current)
        visited.add(current)
        current = frames.get(current, {}).get('ako')

    return chain


def frame_query(concept, slot, value=None):
    """Frame inheritance lookup with local exception priority."""
    for current in frame_chain(concept):
        frame = frames[current]

        if slot == 'can' and 'cannot' in frame:
            if value is None or frame['cannot'] == value:
                return False if value is not None else ('cannot', frame['cannot'])

        if slot in frame:
            stored = frame[slot]
            if value is None:
                return True, stored
            if stored == value:
                return True

    return None if value is not None else 'unknown'


print("Penguin frame chain:", ' -> '.join(frame_chain('penguin')))
print("penguin can fly?", frame_query('penguin', 'can', 'fly'))
print("penguin has skin?", frame_query('penguin', 'has', 'skin'))
print("shark can swim?", frame_query('shark', 'can', 'swim'))

Penguin frame chain: penguin -> bird -> animal
penguin can fly? False
penguin has skin? True
shark can swim? True


### Observation — Frames

Frames make the representation slot-oriented: each concept contains its local fillers and an `ako` parent. The lookup follows `ako` links upward while preserving the same specific-before-general exception behaviour as the triple-based semantic network.

# Exercise 3 — Multiple Parents

The sheet asks to allow multiple `is-a` links and justify the search order. fileciteturn0file0L45-L46

### Search-order decision

For multiple parents, this implementation uses **breadth-first search (BFS)** from the most specific concept. Therefore, all immediate parents are considered before grandparents. If two facts occur at the same inheritance depth, the order in which the parents were stored is used as the tie-breaker.

This is a reasonable choice because it prefers the closest inherited knowledge and avoids arbitrarily jumping through a deeper branch while a nearer parent still has a relevant property.

In [9]:
# Multiple-parent semantic network
multi_kb = [
    ('animal', 'has', 'skin'),
    ('animal', 'needs', 'food'),
    ('bird', 'is-a', 'animal'),
    ('bird', 'can', 'fly'),
    ('bird', 'has', 'feathers'),
    ('fish', 'is-a', 'animal'),
    ('fish', 'can', 'swim'),
    ('flying_animal', 'can', 'fly'),
    ('flying_animal', 'needs', 'wings'),
    ('duck', 'is-a', 'bird'),
    ('duck', 'is-a', 'fish'),
    ('duck', 'can', 'quack'),
    ('duck', 'cannot', 'fly'),
]

multi_parents = {}
multi_facts = {}
multi_exceptions = {}

for subject, relation, obj in multi_kb:
    if relation == 'is-a':
        multi_parents.setdefault(subject, []).append(obj)
    elif relation == 'cannot':
        multi_exceptions.setdefault((subject, 'can'), set()).add(obj)
    else:
        multi_facts.setdefault((subject, relation), set()).add(obj)

print("Multiple parents of duck:", multi_parents['duck'])

Multiple parents of duck: ['bird', 'fish']


In [10]:
from collections import deque

def bfs_chain(node):
    """Return concepts in BFS order from the most specific node."""
    queue = deque([node])
    visited = set()
    order = []

    while queue:
        current = queue.popleft()
        if current in visited:
            continue
        visited.add(current)
        order.append(current)
        for parent in multi_parents.get(current, []):
            if parent not in visited:
                queue.append(parent)

    return order


def multi_query(node, relation, target):
    for current in bfs_chain(node):
        # Specific exceptions always get priority.
        if target in multi_exceptions.get((current, relation), set()):
            return False
        if target in multi_facts.get((current, relation), set()):
            return True
    return None


print("BFS inheritance order for duck:")
print(' -> '.join(bfs_chain('duck')))

print("\nQueries:")
print("duck can fly? ", multi_query('duck', 'can', 'fly'))
print("duck can swim?", multi_query('duck', 'can', 'swim'))
print("duck has skin?", multi_query('duck', 'has', 'skin'))
print("duck can quack?", multi_query('duck', 'can', 'quack'))

BFS inheritance order for duck:
duck -> bird -> fish -> animal

Queries:
duck can fly?  False
duck can swim? True
duck has skin? True
duck can quack? True


# Final Observations and Comments

1. A semantic network represents knowledge declaratively as nodes and labelled relationships.
2. `is-a` links create an inheritance hierarchy, allowing concepts to inherit properties from ancestors.
3. Exceptions must be checked before inherited values at every level; otherwise `penguin can fly` would incorrectly become true from the bird-level default.
4. The ostrich example demonstrates the same default-with-exception mechanism as the penguin example.
5. Frame representation stores the same knowledge as concept dictionaries containing slots, fillers, and an `ako` parent.
6. Multiple inheritance requires a defined conflict-resolution/search strategy. Here, BFS gives priority to concepts closer to the queried node.
7. Because a more specific fact can defeat a previously applicable default, this is an example of **non-monotonic default reasoning**: adding information can change the answer to a query.

These observations directly address the experiment's required comments on exceptions, inheritance, and non-monotonic reasoning. fileciteturn0file0L35-L38

## Conclusion

The experiment successfully implements a semantic network using `(subject, relation, object)` triples, performs inheritance through `is-a` links, handles local exceptions before inherited defaults, represents the same knowledge using frames, and extends inheritance to multiple parents.